# 00 - Ingesta y esquema relacional

## Resumen

Este notebook es el punto de entrada tecnico del proyecto. Su funcion es reconstruir una base SQLite canonica a partir de **LexGLUE / ECtHR Task B** y documentar el esquema relacional que usaran los notebooks posteriores.

El criterio de reproducibilidad del enunciado exige que otra persona pueda reconstruir el trabajo desde el codigo. Aqui la reconstruccion depende de dos piezas: `schema.sql`, que define SQLite, y `project_utils.py`, que materializa los datos siguiendo ese esquema.

Las verificaciones de integridad, la matriz multietiqueta y los ejemplos interpretados quedan en el notebook 01, porque forman parte del EDA y de la lectura metodologica del dataset.

## Indice

1. Reconstruccion desde `schema.sql`.
2. Esquema relacional explicado.
3. Resultado del notebook.


![Esquema especifico generado con Image Gen](artifacts/figures/generated/notebook_00_ingestion_schema_v2.png)

**Lectura del esquema.** La imagen resume la ingesta reproducible: documentos juridicos reales, separacion en parrafos, normalizacion y materializacion en SQLite con comprobaciones de integridad.


In [1]:
from pathlib import Path
import json, sqlite3, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
import project_utils as pu
warnings.filterwarnings('ignore')
pu.configure()
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_colwidth', 180)
print(f'Raiz del proyecto: {pu.ROOT}')
print(f'Base SQLite: {pu.DB}')

Raiz del proyecto: C:\Users\mauro\OneDrive\Escritorio\All\IA_DEFINITIVO\Año3\Semestre2\Aprendizaje_Avanzado\proyecto final
Base SQLite: C:\Users\mauro\OneDrive\Escritorio\All\IA_DEFINITIVO\Año3\Semestre2\Aprendizaje_Avanzado\proyecto final\data\interim\metadata.db


## 1. Reconstruccion desde `schema.sql`

La siguiente celda fuerza la reconstruccion de `metadata.db`. Si se borra la base, este notebook vuelve a crearla leyendo `schema.sql` y cargando LexGLUE desde Hugging Face o desde el cache local.


In [2]:
from pathlib import Path
import project_utils as pu

pu.RAW = Path(r"C:\hf_cache_lexglue")
pu.RAW.mkdir(parents=True, exist_ok=True)

FORCE_REBUILD_FROM_SCHEMA = True
status = pu.materialize_database(force=FORCE_REBUILD_FROM_SCHEMA)

## 2. Esquema relacional explicado

- `cases`: una fila por caso, texto unido y metadatos de longitud.
- `case_paragraphs`: parrafos originales para trazabilidad.
- `articles`: catalogo de las diez etiquetas de ECtHR Task B.
- `case_labels`: relacion multietiqueta caso-articulo.
- `experiment_runs`: configuracion y metricas de experimentos.
- `predictions`: scores y predicciones por caso.
- `explanations`: artefactos de XAI.

La tabla clave es `case_labels`: un caso puede aparecer varias veces, una por articulo positivo. Esta estructura corresponde a la matriz binaria `Y` que se usa en modelado.


In [3]:
with pu.connect_db() as conn:
    display(pd.read_sql_query("SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY type, name", conn))
    for table in ['cases','case_paragraphs','articles','case_labels','experiment_runs','predictions','explanations']:
        print(f'\n--- {table} ---')
        display(pd.read_sql_query(f'PRAGMA table_info({table})', conn))


,name,type
0,articles,table
1,case_labels,table
2,case_paragraphs,table
3,cases,table
4,experiment_runs,table
5,explanations,table
6,predictions,table
7,v_case_label_codes,view
8,v_case_label_summary,view



--- cases ---


,cid,name,type,notnull,dflt_value,pk
0,0,case_id,TEXT,0,None,1
1,1,task,TEXT,1,None,0
2,2,split,TEXT,1,None,0
3,3,year,INTEGER,0,None,0
4,4,text_full,TEXT,1,None,0
5,5,n_paragraphs,INTEGER,1,None,0
6,6,n_tokens,INTEGER,1,None,0



--- case_paragraphs ---


,cid,name,type,notnull,dflt_value,pk
0,0,case_id,TEXT,1,None,1
1,1,paragraph_idx,INTEGER,1,None,2
2,2,paragraph_text,TEXT,1,None,0


...

,cid,name,type,notnull,dflt_value,pk
0,0,run_id,TEXT,0,None,1
1,1,stage,TEXT,1,None,0
2,2,model_name,TEXT,1,None,0
3,3,config_json,TEXT,1,None,0
4,4,metrics_json,TEXT,1,None,0
5,5,created_at,TEXT,1,None,0
6,6,git_commit,TEXT,0,None,0



--- predictions ---


,cid,name,type,notnull,dflt_value,pk
0,0,run_id,TEXT,1,None,1
1,1,case_id,TEXT,1,None,2
2,2,y_true_json,TEXT,1,None,0
3,3,y_pred_json,TEXT,1,None,0
4,4,scores_json,TEXT,0,None,0



--- explanations ---


,cid,name,type,notnull,dflt_value,pk
0,0,run_id,TEXT,1,None,1
1,1,case_id,TEXT,1,None,2
2,2,method,TEXT,1,None,3
3,3,artifact_path,TEXT,0,None,0
4,4,summary_json,TEXT,1,None,0


## 3. Resultado del notebook

Quedan preparados `data/interim/metadata.db` y `artifacts/reports/ingestion_status.json`. Los demas notebooks pueden ejecutarse directamente porque llaman a la misma materializacion si la base falta.

A partir de aqui, el analisis de integridad, la matriz multietiqueta y los ejemplos reales se desarrollan en `01_datos_y_eda.ipynb`.
